In [1]:
import json
import pandas as pd
from collections import Counter
import shap
import pickle

In [2]:
import os
import sys

project_root = os.path.abspath("..")
sys.path.insert(0, project_root)

In [3]:
from data_gathering.utils.util import clean_text
from classification.utils_finetune import load_dataset, split_dataset

In [4]:
with open(project_root + "/params.json", 'r') as f:
    PARAMS = json.load(f)

In [5]:
device = "cuda"

In [6]:
model_id = PARAMS["classi_finetune_model"]
dataset = load_dataset()
print(f"Size of dataset: {len(dataset)}")
labels = list(dataset["label"].unique())

label2id, id2label, train_data, test_data, val_data = split_dataset(dataset, labels, train_size=PARAMS["train_split"], val_size=PARAMS["val_split"])

print(f'Train: {Counter(train_data["label"])}')
print(f'Test: {Counter(test_data["label"])}')
print(f'Val: {Counter(val_data["label"])}')

df = pd.read_csv("data/themes_from_thesis.csv", sep="\t")
primary_texts = [clean_text(x) for x in df["EXCERPT"]]

Cleaning text


100%|██████████| 3067/3067 [00:00<00:00, 180414.72it/s]

Size of dataset: 3067
{'HIGH': 0, 'MEDIUM': 1, 'LOW': 2}
{0: 'HIGH', 1: 'MEDIUM', 2: 'LOW'}
Train: Counter({1: 1156, 2: 883, 0: 414})
Test: Counter({1: 145, 2: 110, 0: 52})
Val: Counter({1: 144, 2: 111, 0: 52})


In [7]:
from xai.utils import predict_proba, get_tokenizer, get_class_contribs, print_attr

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [8]:
# PartitionExplainer is the right choice for transformer tokenizers
tokenizer = get_tokenizer()

explainer = shap.Explainer(
    predict_proba,
    tokenizer,
    output_names=["LOW", "MEDIUM", "HIGH"],  # your class labels
    silent=True
)

# Run on a sample (SHAP is slow — start with 50-100 examples)
print("Val data")
sample_texts_val = list(val_data["text"])[:5]
shap_values_val = explainer(sample_texts_val)
print("Primary texts")
sample_texts_primary = list(primary_texts)[:5]
shap_values_primary = explainer(sample_texts_primary)


Val data
Primary texts


In [9]:
shap_val = shap_values_val.values
data_val = [tokenizer.tokenize(x) for x in sample_texts_val]

shap_primary = shap_values_primary.values
data_primary = [tokenizer.tokenize(x) for x in sample_texts_primary]

In [10]:
with open(f"shap_values/shap_val.pkl", "wb") as f:
        pickle.dump(shap_val, f)

with open(f"shap_values/shap_primary.pkl", "wb") as f:
        pickle.dump(shap_primary, f)

In [ ]:
val_contribs, val_contexts = get_class_contribs(data_val, shap_val, group=4)
primary_contribs, primary_contexts = get_class_contribs(data_primary, shap_primary, group=4)

In [16]:
print_attr(val_contribs, val_contexts, id2label, attr="positive")


Top words for class HIGH:

Tolosa is situated upon: 0.0784
    [[Tolosa is situated upon]] the narrowest part of the isthmus which separates

was previously called Hyle: 0.0288
    a city The home town of Zeno It [[was previously called Hyle]] being a colony of the Phocaeans a frugal

the ocean from the: 0.0181
    the narrowest part of the isthmus which separates [[the ocean from the]] sea of Narbonne the breadth of the according

Syrtis islands and the: 0.0164
    passed the Eridanus the [[Syrtis islands and the]] Sirens while Orpheus played his lyre only Butes

coast of the mainland: 0.0144
    the Kyrenians made their settlement and on the [[coast of the mainland]] there is Port Menelaos and Aziris where the

off the coast the: 0.0138
    Aphrodisias In the space within this limit lies [[off the coast the]] island of Platea where the Kyrenians made their

the isthmus which separates: 0.0068
    Tolosa is situated upon the narrowest part of [[the isthmus which separates]] the ocean

In [14]:
print_attr(val_contribs, val_contexts, id2label, attr="negative")


Top words for class HIGH:

Narbonne the breadth: -0.0863
    the ocean from the sea of [[Narbonne the breadth]] of the according to Posidonius being

Platea where the: -0.0730
    off the coast the island of [[Platea where the]] Kyrenians made their settlement and on

to Posidonius being: -0.0677
    Narbonne the breadth of the according [[to Posidonius being]] less

of Zeno It: -0.0618
    of a city The home town [[of Zeno It]] was previously called Hyle being a

of Aphrodisias In: -0.0561
    West as far as the island [[of Aphrodisias In]] the space within this limit lies

nourish only: -0.0329
    Phocaeans a frugal city which could [[nourish only]]

to King Alcinous: -0.0311
    of Helios and came to Phaeacia [[to King Alcinous]] Of the Colchians who

city which could: -0.0299
    colony of the Phocaeans a frugal [[city which could]] nourish only

Scylla and Thetis: -0.0292
    passed Charybdis the blue cliffs of [[Scylla and Thetis]] guided them through at the request

dreadful a

In [ ]:
print_attr(primary_contribs, primary_contexts, id2label, attr="positive")


Top words for class HIGH:

when the sun: 0.0246
    have arrived on a harborless coast [[when the sun]] is sinking into night

to Delphi they: 0.0103
    and of the deputies who came [[to Delphi they]] corrupted some with money one of

You know of: 0.0066
    [[You know of]] your own knowledge and have no

full sail and: 0.0055
    the ships entered the harbour under [[full sail and]] again crammed the storehouses with grain

Londinium intending there: 0.0047
    lay opposite and went on to [[Londinium intending there]] to form his plans according to

deputies who came: 0.0040
    into the harbor and of the [[deputies who came]] to Delphi they corrupted some with

is sinking into: 0.0024
    a harborless coast when the sun [[is sinking into]] night

the harbour under: 0.0024
    southern breeze and the ships entered [[the harbour under]] full sail and again crammed the

and then sailed: 0.0022
    he waited for a favourable breeze [[and then sailed]] to Rutupiae which lay opposite and

In [ ]:
print_attr(primary_contribs, primary_contexts, id2label, attr="negative")


Top words for class HIGH:

to Rome from: -0.0530
    the divine power that gave increase [[to Rome from]] its cradle and promised that it

the sacred harbor: -0.0303
    how they are making money from [[the sacred harbor]]

when the securing: -0.0241
    so speedy nor yet in anchoring [[when the securing]] cables must be brought ashore and

Ostia a calm: -0.0241
    temple of Castor and Pollux at [[Ostia a calm]] smoothed the sea the wind changed

and have no: -0.0228
    You know of your own knowledge [[and have no]] need of other witness how these

that was dedicate: -0.0227
    rebuilt the walls of the harbor [[that was dedicate]] and accursed and settled there and

shepherds of ships: -0.0225
    brought ashore and even at anchorage [[shepherds of ships]] do not feel immediately secure above

brought ashore and: -0.0204
    when the securing cables must be [[brought ashore and]] even at anchorage shepherds of ships

witness how these: -0.0200
    and have no need of other [[witnes